In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
batch1_df=spark.table("hospital_appointments.bronze.batch1_raw_data")

In [0]:
batch1_df.printSchema()

In [0]:
display(batch1_df)

In [0]:
file_path='/Volumes/hospital_appointments/source/input_raw_files/Hospital_Appointments_Incremetal.csv'

In [0]:
batch2_df= (
    spark.read.format("csv")
    .option("header","true")
    .option("inferSchema","true").load(file_path)
)
display(batch2_df)

In [0]:
batch2_df=(
   batch2_df
    .withColumn("source_file",lit(file_path))
    .withColumn("batch_id",lit("BATCH_02"))
    .withColumn("ingestion_timestamp",current_timestamp())
)
display(batch2_df)


In [0]:
batch1_df.printSchema()



In [0]:
batch2_df.printSchema()

In [0]:
from pyspark.sql.functions import *

batch1_df = batch1_df.withColumn(
    "appointment_date",
    coalesce(
        try_to_date(col("appointment_date"), "yyyy-MM-dd"),
        try_to_date(col("appointment_date"), "dd-MM-yyyy"),
        try_to_date(col("appointment_date"), "MM/dd/yyyy"),
        try_to_date(col("appointment_date"), "M/d/yyyy")
    )
)

In [0]:
batch1_df.printSchema()

In [0]:
all_raw_df = batch1_df.unionByName(batch2_df)



In [0]:
all_raw_df.printSchema()

In [0]:
display(all_raw_df)

In [0]:
all_raw_df.write \
.mode("overwrite") \
.saveAsTable("hospital_appointments.bronze.all_raw_data")

In [0]:
display(spark.table("hospital_appointments.bronze.all_raw_data"))